# Forecast-period bias metrics

Bias, RMSE, spread and CRPS maps for the atmospheric hindcasts (`fc`) from `util.bias_metrics`, with optional bootstrap confidence intervals that hatch significant biases. This is the forecast counterpart of `analysis_atm_bias/01_compute_and_plot_bias_metrics.ipynb`; its caches and figures are kept separate from the analysis-period workflow.

Cases: **S0** hindcasts start from the Dec 2011 analyses and cover Jan-Feb 2012; **S1** hindcasts start from the May 2012 analyses and cover Jun-Jul 2012. Each case compares hindcasts initialized from CTRL, CAPT and EAM-DART states.

## 1. Settings

In [ ]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/compyfs/zhan391/acme_init/Observations")  # observations (shared archive)
workflow_name = "fcst_atm_bias"

component = "atm"
run_segment = "fc"
# Forecast cases: season -> experiments compared together (same period).
cases = {
    "S0": ("CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"),  # 201201-201202
    "S1": ("CTRL10-S1", "CAPT10-S1", "DART40-S1"),  # 201206-201207
}
variables = ("FLUT", "PRECT", "TMQ", "TS", "TREFHT", "PSL")
plot_metrics = ("bias", "rmse", "spread", "crps")
n_bootstrap = 100  # 0 skips confidence intervals (no bias hatching)
random_seed = 42
lat_range = (-90, 90)
lon_range = (-180, 180)
max_members = None  # e.g. 2 for a quick test
# Existing caches are reused unless they were built from another model source;
# force_compute rebuilds them regardless.
force_compute = False
show = True
FIGURE_DPI = 200

## 2. Project setup

In [ ]:
import sys
import matplotlib.pyplot as plt

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from IPython.display import Image, display
from util.bias_metrics import METRICS, available_experiments, plot_cached_bias_metrics, run_bias_metrics

data_dir, figure_dir = workflow_output_dirs(
    workflow_name, "bias_metrics", output_root=DIAGNOSTIC_OUTPUT_ROOT
)
unknown = set(plot_metrics) - set(METRICS)
if unknown:
    raise ValueError(f"Unknown metrics {sorted(unknown)}; choose from {list(METRICS)}")
for season in cases:
    print(f"{season} available experiments:", available_experiments(season, run_segment))

## 3. Compute or reuse bias metrics

In [ ]:
for season, experiments in cases.items():
    run_bias_metrics(
        data_root=INPUT_DATA_ROOT, reference_root=OBSERVATION_ROOT,
        cache_dir=data_dir, season=season, run_segment=run_segment,
        component=component, experiments=experiments, variables=variables,
        lat_range=lat_range, lon_range=lon_range, n_bootstrap=n_bootstrap,
        random_seed=random_seed, max_members=max_members,
        force_compute=force_compute,
    )

## 4. Plot, save, and show

In [ ]:
for season, experiments in cases.items():
    figures = plot_cached_bias_metrics(
        data_root=INPUT_DATA_ROOT, cache_dir=data_dir, figure_dir=figure_dir,
        season=season, run_segment=run_segment, component=component,
        experiments=experiments, variables=variables, metrics=plot_metrics,
        dpi=FIGURE_DPI,
    )
    for output in figures:
        print(f"[FIGURE] {output}")
        if show:
            display(Image(filename=str(output)))

## Notes: variables and metrics

| Variable | Reference | Units |
|---|---|---|
| `FLUT` | CERES-OAFlux | W m-2 |
| `PRECT` | GPCP | mm day-1 |
| `TMQ` | ERA5 | kg m-2 |
| `TS` | ERA5 | degC |
| `TREFHT` | ERA5 | degC |
| `PSL` | ERA5 | hPa |

| Metric | Meaning |
|---|---|
| `bias` | ensemble-mean model minus reference, averaged over the forecast period |
| `rmse` | root-mean-square member error over ensemble and time |
| `spread` | ensemble standard deviation averaged over time |
| `crps` | continuous ranked probability score averaged over time |

Only experiments with a configured `fc` run are available (printed in the project-setup cell). Caches: one request-keyed NetCDF per experiment/variable/period under `bias_metrics/`.